# 05. 카메라 영상과 로봇 상태를 VLA 입력으로 준비하기

**목적: 가상 환경의 관측이 어떤 변환을 거쳐 정책 입력이 되는지 이해하고, 현재 상태와 앞으로 실행할 행동의 차이를 설명합니다.**

예상 시간 60분. [전체 안내](README.md).

04에서는 그림을 보고 “어떤 물체가 어디에 있는가?”를 확인했습니다. 여기서는 같은 장면을 **모델에 전달할 숫자 배열** 로 살펴봅니다. 사람이 사진을 이해하는 것과 달리 모델은 정해진 카메라 순서·배열 축·값 범위·상태 순서로 입력을 받아야 합니다.

예를 들어 외부 카메라와 손목 카메라가 모두 정상적인 사진이어도 이름을 서로 바꾸면 모델에 전달하는 의미가 달라집니다. 위치와 회전도 각각 숫자 3개이지만 순서를 교환하면 잘못된 로봇 상태를 전달합니다. 그래서 **배열의 크기뿐 아니라 숫자의 의미와 변환 순서** 를 확인합니다.

이 단원은 입력 배열 읽기와 영상 변환을 직접 실행하고, 전체 전처리와 행동 출력을 설명으로 연결합니다. 실제 정책 추론은 06, 행동을 적용하는 폐루프는 07에서 수행합니다.


## 05의 화면: 저장 영상 확인

05는 저장된 관측 배열을 분석하므로 새 MuJoCo GUI를 열지 않습니다. 배열 읽기 셀에서 같은 실행의 두 PNG를 표시합니다. 결과가 없을 때의 inspect 생성도 이 단원에서는 창 없이 수행합니다. 실제 추론·실행 GUI는 06·07에서 사용합니다.


## 전체 프로세스에서 지금 어디에 있나요?

![가상 환경의 관측에서 VLA 입력·행동·재관측까지의 흐름](../교재그림/vla-observation-action-flow.svg)

그림의 **파란 단계는 05에서 직접 실행** , 보라색은 실제 runner의 입력 준비 중 설명으로 살펴보는 부분, 초록색·주황색은 06·07에서 이어지는 부분입니다. 05의 영상 변환은 저장된 영상 한 장으로 같은 연산을 확인하는 예제이며, 전체 정책 입력을 완성하는 코드는 아닙니다.

1. **04 — 환경 초기화:** LIBERO의 과제와 초기 배치를 정하고 reset합니다. 두 영상·로봇 상태·언어 지시를 얻습니다.
2. **05 — 저장 배열 확인:** `initial-observation.npz`에서 두 영상과 상태 8개를 읽습니다. 언어 지시는 별도로 `result.json`의 `task`에 기록됩니다.
3. **05 — 영상 변환 실습:** 방향을 맞추고, 배열 축과 자료형·값 범위를 바꿉니다. 실제 `make_batch`는 두 카메라 모두에 이 처리를 적용합니다.
4. **실제 runner의 나머지 입력 준비:** 저장된 processor가 카메라 이름을 바꾸고 배치 축을 추가하며, 지시를 토큰으로 변환하고 CUDA로 옮긴 뒤 상태를 정규화합니다. 05에서는 코드와 설정을 읽어 역할을 이해합니다.
5. **06 — 정책 추론과 후처리:** 준비한 입력에서 행동 묶음을 예측하고 학습 정규화를 되돌립니다. 한 번의 추론 결과를 확인하며 환경에 행동을 적용하지 않습니다.
6. **07 — 실행과 재관측:** 기본적으로 앞 10개의 행동을 하나씩 적용하고, 계속 진행할 경우 새 관측으로 다시 추론합니다. 각 step에서 성공·종료를 확인합니다.

**파일 연결과 실시간 연결은 구분합니다.** 05는 04의 저장 관측을 읽는 연습입니다. 실제 06·07 runner는 자신의 환경을 reset한 관측으로 입력을 만들며, 매번 04의 PNG를 읽어 추론하지 않습니다. 07의 다음 묶음은 행동 후 새 관측을 사용합니다.


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [5일차 실행 안내](README.md)를 참고합니다.

**5일차 환경:** `nepes-course:physical-ai-vla-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 같은 날에는 준비된 개인 VLA 환경을 유지합니다.

**실행 전 확인 — 컨테이너 내부 Terminal:**

```bash
cd /workspace
/opt/course/bin/python 10.VLA/scripts/preflight.py
```

Python 커널도 `/opt/course/bin/python`을 사용합니다. 강사 배포 `Data/models/smolvla-libero-probe`(LIBERO용 정책 revision `31d453f7edd78c839a8bbc39744a292686daf0de`), `Data/cache/smolvla-probe/huggingface`, `Data/cache/smolvla-probe/libero`와 LIBERO 자산이 필요합니다. 파일 누락은 호스트 개인 Data의 마운트·배포본을 확인합니다. 수강생이 `pip install`, Docker build, 모델 다운로드를 하지 않습니다. CUDA/EGL 문제는 전용 이미지·NVIDIA runtime과 다른 GPU 작업을 확인합니다.

**실행과 저장:** 04부터 08까지의 노트북의 `SESSION = 'session-01'`은 같은 회차이며 결과는 `Data/logs/day5/session-01/<실행이름>/`에 남습니다. 새 비교 전체는 모든 후속 노트북의 SESSION을 함께 `session-02` 등으로 바꿉니다. `run_lesson`은 동일 조건의 `completed` 결과를 읽고 없을 때 실행합니다. 미완료·조건 불일치는 기존 파일을 삭제하거나 성공값으로 바꾸지 말고 새 회차/실행 이름을 사용합니다. JSON은 있는데 필요한 PNG/NPZ/NPY/영상이 없으면 부분 손상으로 보고 새 회차에서 다시 생성합니다.

**한 번에 한 정책 프로세스만 실행합니다.** 이전 실행이 끝난 뒤 다음 셀을 실행하고 다른 노트북의 infer/rollout을 동시에 시작하지 않습니다. 실행 중단 후에는 해당 Terminal이나 실행 로그에서 자식 프로세스까지 종료됐는지 확인합니다. `status=completed`는 절차 완료, rollout의 `success=true`는 환경 과제 성공이며 inspect/infer의 `success=false`는 아직 작업을 하지 않았다는 뜻입니다.

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **예제·TODO·검증** → [5일차 슬라이드 32쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s028) · [배포 PDF 32쪽](../슬라이드/5일차.pdf#page=32)에서 37쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이전 자료의 위치·필요한 내용·없는 파일 복구

[04_environment.ipynb의 「예제」](04_environment.ipynb)가 만든 `Data/logs/day5/<SESSION>/inspect-01/initial-observation.npz`를 읽습니다. 그 파일에는 `image,image2` 두 RGB uint8 `(256,256,3)` 배열과 `state` float `(8,)`가 있습니다. 상태의 앞3개는 끝단 위치, 다음3개는 quaternion을 axis-angle로 바꾼 회전, 마지막2개는 그리퍼 관절입니다.

아래 첫 설정 셀에서 SESSION/RUNS/run_lesson을 다시 만들고, 배열 읽기 셀은 `run_lesson('inspect','inspect-01')`로 동일 조건 결과를 확인하거나 **없으면 정책 없이 reset하여 생성** 합니다. 04 노트북을 다시 열 필요가 없습니다. JSON만 있고 배열이 없거나 조건이 다르면 시작 절의 새 SESSION 복구를 따릅니다.

상세 변환 위치는 [scripts/course_runtime.py의 make_batch](scripts/course_runtime.py)와 `/workspace/Data/models/smolvla-libero-probe/policy_preprocessor.json`입니다. 핵심은 raw 영상을 가로·세로 한 번 반전 → HWC에서 CHW → float32/255, image→camera1·image2→camera2 이름 변경, 상태8에 저장 정규화 통계 적용입니다. 아래 셀의 변환은 raw NPZ에서 시작하고 표시용 PNG에 다시 반전을 적용하지 않습니다.

## 개념

상태8은 끝단 위치3 + axis-angle 회전3 + 그리퍼 관절2입니다. config의 오래된 상태 shape=6과 달리 실제 데이터·정규화 통계는 8입니다. 자르거나 가짜 값을 넣지 않습니다. 코드가 통계 차원을 검사합니다.

두 RGB 영상을 가로·세로 반전하고 HWC uint8→CHW float `[0,1]`로 바꿉니다. 제공 processor가 image/image2를 camera1/camera2로 변경하고 상태를 정규화합니다. camera3는 기존 모델의 누락 처리 경로를 사용하며 필수 영상 둘이 없으면 오류입니다.

행동7은 정규화된 끝단 이동3·회전3·그리퍼1입니다. 절대 좌표나 거북이 속도가 아닙니다. 제공 postprocessor가 학습 정규화를 되돌리고 환경 제어기로 전달합니다.

### 단계별 활동

진행: 저장 배열 읽기15분 → RGB/축/형상 확인15분 → 상태 구성15분 → 정규화·행동 계약15분. make_batch 함수와 policy_preprocessor.json을 함께 읽어 각 변환 위치를 찾으세요. 배열의 shape와 값 범위는 서로 다른 정보입니다. 위치3과 회전3을 서로 교환하면 차원 검사는 통과해도 의미가 틀린다는 점을 토의합니다.

In [ ]:
from pathlib import Path
import sys, subprocess, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '10.VLA').is_dir())
SCRIPTS = ROOT / '10.VLA/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, lesson_runs, run_or_read_lesson
# 04~08 모두 같은 회차를 사용합니다. 전체 비교를 새로 할 때 session-02 등으로 함께 변경하세요.
SESSION = 'session-01'
RUNS = lesson_runs(DATA, SESSION)
RUNS.mkdir(parents=True, exist_ok=True)
def run_lesson(mode, name, seed=7, execute=10, steps=280, initial_state=0):
    return run_or_read_lesson(SCRIPTS, RUNS, mode, name, seed, execute, steps, initial_state)


## 예제

## 첫 배열 읽기 셀: 무엇을 확인하나요?

`inspection = run_lesson('inspect', 'inspect-01')`은 현재 SESSION의 기준 결과를 확인합니다. 같은 조건의 완료 결과가 있으면 읽고, 없다면 정책을 로드하지 않고 생성합니다. `obs = np.load(...)`는 그 실행에서 저장한 숫자 배열을 엽니다.

| 항목 | 저장된 형태 | 뜻 | 확인 이유 |
|---|---|---|---|
| `image` | `(256, 256, 3)`, `uint8` | 첫 카메라의 RGB 영상 | 높이·너비·색상 채널과 자료형이 맞는지 |
| `image2` | `(256, 256, 3)`, `uint8` | 다른 시점의 RGB 영상 | 두 번째 필수 관측이 실제로 있는지 |
| `state` | `(8,)`, 실수 배열 | 같은 시점의 로봇 상태 | 값의 개수와 유한한 수인지 |
| `inspection['task']` | 문자열 | 어떤 작업을 수행할지 나타내는 지시 | 동일 영상에서도 목표에 따라 행동이 달라지므로 |

`assert`는 예상 조건을 검사합니다. 통과했다는 것은 해당 형상·자료형·값 조건이 맞다는 뜻이며, 사진의 내용과 카메라 역할까지 자동으로 검증했다는 뜻은 아닙니다.

### 상태 8개를 읽는 법

```text
state = [ x, y, z,  rx, ry, rz,  q_left, q_right ]
          위치 3      회전 3        그리퍼 관절 2
```

- `state[:3]`: 로봇 끝단이 현재 어디에 있는지 나타냅니다.
- `state[3:6]`: 현재 끝단의 방향을 axis-angle 형식의 3개 값으로 나타냅니다. 각각을 단순한 roll·pitch·yaw 각도로 해석하지 않습니다.
- `state[6:]`: 두 그리퍼 관절의 현재 위치입니다. 하나의 열기/닫기 명령과는 다릅니다.

환경의 원래 회전은 quaternion입니다. 수업의 `make_batch`가 axis-angle로 변환한 뒤 위치·그리퍼 관절과 이어 붙입니다. **저장된 NPZ의 상태는 이미 이 변환을 거친 8개 값** 이므로 여기서 quaternion 변환을 다시 하지 않습니다. 다만 정책용 평균·표준편차 정규화는 아직 적용하지 않은 값입니다.


In [ ]:
import numpy as np
# 04의 결과가 없으면 여기서 정책 로드 없이 reset하고 같은 입력 파일을 생성합니다.
inspection = run_lesson('inspect', 'inspect-01')
obs = np.load(RUNS/'inspect-01/initial-observation.npz')
print({key: obs[key].shape for key in obs.files})
assert obs['state'].shape == (8,)
for camera in ('image', 'image2'):
    assert obs[camera].shape == (256,256,3)
    assert obs[camera].dtype == np.uint8
assert np.isfinite(obs['state']).all()
print('위치:', obs['state'][:3], '회전:', obs['state'][3:6], '그리퍼:', obs['state'][6:])
from IPython.display import Image, display
for camera in ('image', 'image2'):
    display(Image(filename=str(RUNS/'inspect-01'/f'{camera}.png')))


### 입력과 실행 흐름 확인

## 영상 변환 셀: 한 줄을 네 연산으로 나누어 읽기

<!-- procedure:lesson-41-3ed9da:start -->
![저장 관측을 모델 입력으로 읽기](../교재그림/vla-observation-action-flow.svg)

05는 저장 배열의 변환을 이해합니다. 한 장 변환만으로 실제 정책 입력 전체를 구성한 것은 아닙니다.
<!-- procedure:lesson-41-3ed9da:end -->


아래 셀은 `obs['image']` 한 장을 사용합니다. 현재 출력문은 원본 형상·자료형, 변환 후 형상·최솟값·최댓값을 보여 줍니다.

| 연산 | 하는 일 | 하지 않는 일 |
|---|---|---|
| `raw[::-1, ::-1].copy()` | 높이·너비 방향을 모두 뒤집어 LIBERO 어댑터의 방향에 맞춤 | RGB 색상 순서는 바꾸지 않음 |
| `.transpose(2, 0, 1)` | HWC → CHW로 배열 축 순서를 변경 | 사진을 회전하거나 확대하지 않음 |
| `.astype('float32')` | 정수 픽셀을 실수로 표현 | 이 단계만으로 0–1 범위가 되지는 않음 |
| `/ 255` | 0–255 범위를 0–1 범위로 변환 | 데이터셋 평균·표준편차 정규화와는 다른 연산 |

가령 **설명용 픽셀** `[0, 128, 255]`는 나누기 후 약 `[0.0, 0.502, 1.0]`이 됩니다. 실제 영상에서 이 값이 관측됐다는 뜻은 아닙니다. 영상에 완전한 검정·흰색이 없으면 출력 최솟값·최댓값이 정확히 0과 1일 필요도 없습니다.

`(256, 256, 3)`은 256×256개의 위치마다 RGB 값 3개가 있다는 뜻입니다. `(3, 256, 256)`은 같은 값을 색상 채널별 평면으로 묶었다는 뜻입니다. 정보량은 그대로이며 축 배치만 달라집니다.

**원본 NPZ에서 한 번만 변환합니다.** 04에서 본 PNG는 이미 표시 방향을 맞춰 저장한 그림입니다. 그 PNG에 같은 반전을 다시 적용하면 방향이 되돌아갑니다. 이 셀의 `converted`는 확인용 NumPy 배열이며 실제 모델 추론을 호출하지 않습니다.


In [ ]:
raw = obs['image']
converted = raw[::-1, ::-1].copy().transpose(2,0,1).astype('float32')/255
print(raw.shape, raw.dtype, converted.shape, converted.min(), converted.max())
assert converted.shape == (3,256,256)

## 실제 정책 입력에는 무엇이 더 필요한가요?

실제 runner는 `pre(make_batch(obs, task))`를 호출합니다. `make_batch`가 두 영상·상태 벡터·지시를 모으고, 배포된 `policy_preprocessor.json`의 처리 순서가 이어집니다.

| 처리 | 이번 실행에서의 역할 |
|---|---|
| 카메라 이름 매핑 | `image` → `camera1`, `image2` → `camera2`; 학습 때의 입력 이름과 대응 |
| 배치 축 추가 | 한 관측도 여러 관측을 처리하는 인터페이스에 맞춰 묶음 크기 1로 표현 |
| 언어 지시 처리 | 문자열을 tokenizer로 토큰 ID와 마스크로 바꾸어 모델에 전달 |
| 장치 이동 | 텐서가 추론에 사용하는 CUDA 장치에 놓이도록 준비 |
| 상태 정규화 | 학습 자료의 저장된 평균·표준편차로 상태 8개 각각의 척도를 맞춤 |

상태 정규화는 개념적으로 `(현재 값 - 학습 평균) / 학습 표준편차`입니다. 현재 샘플 8개의 평균을 임의로 빼거나, 모든 성분을 같은 단위로 취급하는 방식이 아닙니다. 이번 processor 설정에서 영상 정규화 방식은 `IDENTITY`이므로 이미 255로 나눈 영상에 임의의 평균·표준편차 처리를 추가하지 않습니다.

설정 일부에 남은 상태 길이 6이라는 메타데이터만 보고 실제 값을 자르지 않습니다. 수업 어댑터는 실제 상태와 배포된 정규화 통계가 8개인지 검사합니다. 이 설명은 **이번 체크포인트와 수업 코드의 조합** 에 대한 것이며 모든 VLA가 상태 8개를 사용한다는 뜻은 아닙니다.

## 출력 행동 7개는 상태 8개와 어떻게 다른가요?

| 구분 | 상태 입력 | 행동 출력 |
|---|---|---|
| 질문 | 지금 어떤 자세인가? | 다음에 어떻게 움직일 것인가? |
| 구성 | 위치 3 + 회전 3 + 그리퍼 관절 2 | 끝단 이동 3 + 회전 3 + 그리퍼 제어 1 |
| 역할 | 현재 상태를 정책에 알려 줌 | 환경의 제어기에 전달할 명령 |
| 주의 | 실제 관측 순서와 단위를 보존 | 절대 목표 좌표나 로봇의 관절각 7개로 해석하지 않음 |

그리퍼는 현재 두 관절의 상태를 관측하면서도, 동작은 하나의 열기/닫기 제어값으로 명령할 수 있습니다. 따라서 **입력 상태와 출력 행동의 길이가 같을 필요가 없습니다.** 정책은 상태 벡터에서 숫자 하나를 지워 행동을 만드는 것이 아니라, 영상·지시·상태를 함께 사용해 행동을 예측합니다.

06의 실제 경로는 `policy.predict_action_chunk(batch)` 다음에 `post(...)`를 적용합니다. 후처리는 학습 정규화를 되돌려 환경이 사용할 행동 값으로 복원합니다. 저장되는 묶음은 이 후처리를 거친 `50×7` 배열입니다. 07은 각 행동의 값을 `[-1, 1]`로 제한해 환경의 제어기에 전달합니다. **학습 정규화를 되돌린 값도 환경 제어기의 정규화된 명령일 수 있으므로** , 숫자 0.1을 곧바로 0.1m 이동으로 읽지 않습니다.

행동이 적용되면 물리 상태와 카메라 영상이 바뀝니다. 계속 진행할 때 이 새 관측으로 다시 입력을 구성하는 것이 폐루프입니다.


## TODO

상태를 6차원으로 자르면 어떤 정보가 없어지는지 설명하세요. 두 카메라 중 하나를 0으로 채우는 대신 중단하는 이유를 적으세요.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**작성 위치와 출력:** 바로 아래 코드 셀의 `notes = {}`에 관찰 답안을 dict로 작성하고 노트북을 저장합니다. 새 정책 함수를 구현하는 TODO가 아닙니다. 코드 검토만 할 때는 아래 형식의 의미만 읽고, 실제로 관찰하지 않은 값은 `None`/`미측정`으로 남깁니다. notes 출력은 파일 저장이 아니므로 답안은 셀에 직접 적고 노트북 자체를 저장하세요.
**입력 자료:** inspect-01/initial-observation.npz와 make_batch·processor 설명.
**권장 답안 형식** (key 이름은 기록용 권장안이며 자동 채점 schema가 아닙니다):
```python
notes = {'state_parts': [3, 3, 2],
 'action_parts': [3, 3, 1],
 'lost_if_state6': '없어지는 두 성분과 정규화에 미치는 영향',
 'missing_camera_policy': '입력 누락에서 중단해야 하는 이유',
 'raw_to_model': '방향/축/dtype/범위 변환 순서'}
```
**진행 힌트:** state[:3],state[3:6],state[6:]의 의미를 대응 → 이미지 HWC/CHW와 uint8/float를 구분 → 상태8과 행동7이 왜 다른지 적습니다. 영상을 0으로 채우거나 상태를 자르는 코드를 작성하는 문제가 아니라 잘못된 수정을 설명하는 문제입니다.
**강의 멘트:** “배열 길이만 맞춰서는 입력 의미를 복구할 수 없습니다. 누락된 관측을 가짜 값으로 채우지 않습니다.”
**완료 확인:** 각 항목에 근거와 이유가 있고 실제 관찰/실측 값과 개념 설명을 구분했는지 확인합니다. 정답 문단의 수치는 관찰하지 않은 자기 실행 결과를 대신하지 않습니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 관찰 결과/실험 설정을 여기에 기록하세요.
notes = {}
notes

## 검증과 정답·복구

그리퍼 관절2가 없어지고 저장된 정규화 통계와 맞지 않습니다. 가짜 영상은 학습 입력 분포와 달라 결과 의미를 보장하기 어렵습니다. 준비된 관측 계약을 지킵니다.

파일·캐시·CUDA 오류는 [복구 안내](README.md#복구)와 이 노트북 시작 절에 있습니다. 필요한 복구는 전용 이미지·/opt/course 커널 확인, Data 마운트·배포 모델/캐시 존재 확인, 다른 GPU 작업 종료 후 한 프로세스 실행입니다. 누락 파일을 다운로드하거나 상태8을 잘라 우회하지 않습니다. CLI는 노트북과 같은 실험의 대체 실행입니다. 같은 조건의 완료 결과는 다시 읽습니다. 새 실험은 SESSION과 후속 노트북의 회차를 함께 바꾸세요. 실험 출력은 삭제하지 않습니다.

### 상세 정답: 이유와 확인 근거를 함께 쓰기

1. **상태를 6개로 자르면?** `state[6:]`에 있던 그리퍼 관절 2개를 잃습니다. 모델은 손이 현재 얼마나 벌어졌는지에 대한 입력을 잃고, 저장된 8차원 정규화 통계와도 맞지 않습니다. 오류를 없애려고 배열을 자르는 것은 올바른 복구가 아닙니다.
2. **필수 카메라가 없을 때 왜 중단하나요?** 0으로 채운 배열은 모양만 맞을 뿐 실제 장면의 관측이 아닙니다. 코드가 실행되더라도 관측 누락을 숨기게 됩니다. 이번 수업은 두 필수 카메라를 확인하고 누락 원인을 해결합니다.
3. **같은 크기의 카메라를 바꾸면?** 외부와 손목 시점이 학습 때 연결된 이름과 달라져 물체·로봇의 관계를 잘못 해석할 수 있습니다. shape 검사만으로는 이 문제를 찾지 못합니다.
4. **영상 방향과 축 변환은 어떻게 다른가요?** 방향 반전은 픽셀의 공간 위치를 바꾸고, HWC→CHW는 같은 픽셀 값을 메모리의 어떤 축으로 해석할지 바꿉니다. `/255`는 값의 척도를 바꿉니다. 서로 대신할 수 없는 연산입니다.
5. **상태 8개와 행동 7개가 다른 이유는?** 현재 자세의 관측과 다음 움직임의 제어는 서로 다른 표현입니다. 두 그리퍼 관절 상태를 하나의 그리퍼 제어 명령과 혼동하지 않습니다.

**완료 점검:** 아래 TODO의 `notes`에 변환 순서와 상태·행동 구성, 잘못된 입력을 임의로 보정하면 안 되는 이유를 자신의 말로 적습니다. 실제 확인한 배열 형상·자료형·범위는 별도로 기록합니다. 설명용 수치와 실제 측정값을 구분하며 기존 답안을 정답 예시로 덮어쓰지 않습니다.


### 정답 해설 — 입력부터 결과까지

**8과 7을 분해하기:** 관측 상태8은 위치3+회전3+그리퍼 관절2입니다. 행동7은 이동3+회전3+그리퍼1이므로 같은 구성이나 단위가 아닙니다. 상태를 6으로 자르면 손가락 관절 정보가 사라지고 저장된 정규화 통계와도 맞지 않습니다.

영상 `(256,256,3)` uint8을 `(3,256,256)` float로 바꾸고 255로 나누면 값 범위가 0–1입니다. 축 변경·방향 반전·정규화는 각각 다른 연산입니다. 전처리를 두 번 적용하지 않도록 실제 processor의 책임을 확인합니다.

**확인 질문:** 두 camera의 shape가 같더라도 이름을 서로 바꾸면 왜 정책 입력의 의미가 달라지나요?

**적용·재확인:** 관찰 답안은 자신의 실제 파일·조건으로 작성합니다. 완료된 동일 조건 결과는 다시 읽고, 새 실험은 새 실행 이름을 사용합니다. `SESSION`을 바꾸면 후속 노트북도 맞춥니다. 형상 검사·실행 완료·작업 성공을 따로 확인하며, 예시 수치를 자기 측정값으로 복사하지 않습니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 영상 2개·상태 8차원·행동 7차원의 계약 기록

[5일차 슬라이드 32쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s028) · [배포 PDF 32쪽](../슬라이드/5일차.pdf#page=32)로 돌아가 **32–37쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

